In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [2]:
event_log_name = "bpic13"
log_path = f"./.out/eventlogs/{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1461 [00:00<?, ?it/s]

In [3]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [4]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
# if True:
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

File bpic13-0.3-1.decl does not exist, running discovery...
Computing discovery ...
Total constraints discovered: 202


In [5]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


File bpic13_5000_conformance_results.pkl does not exist, running conformance checking...
Conformance checking results saved to bpic13_5000_conformance_results.pkl


In [6]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


/tmp/ipykernel_4129221/1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [10]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
# filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.3].sort_values(by='confidence', ascending=False)
print(f"Filtered Metrics DataFrame: {event_log_name}")
display(filtered_metrics_df)

Filtered Metrics DataFrame: bpic13


,support,confidence
"Not Chain Precedence[Queued+Awaiting Assignment, Accepted+Wait] | |",0.259411,0.784679
"Responded Existence[Accepted+Wait, Completed+Closed] | |",0.259411,0.784679
"Not Chain Precedence[Completed+Closed, Accepted+Wait] | |",0.259411,0.784679
"Response[Accepted+Wait, Completed+Closed] | |",0.259411,0.784679
"Not Precedence[Completed+Closed, Accepted+Wait] | |",0.255305,0.772257
...,...,...
Exactly1[Accepted+Wait] | |,0.203970,0.000000
Existence1[Accepted+Wait] | |,0.259411,0.000000
Existence1[Accepted+Assigned] | |,0.285421,0.000000
"Exclusive Choice[Accepted+Assigned, Queued+Awaiting Assignment] | |",0.260780,0.000000


In [8]:
raise KeyboardInterrupt("\nStopping execution after displaying filtered metrics DataFrame.\nChoose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.\nThen run the cells below again to see the results of the selected constraints.")

KeyboardInterrupt: 
Stopping execution after displaying filtered metrics DataFrame.
Choose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.
Then run the cells below again to see the results of the selected constraints.

# Constraints with low support and high confidence
1. Responded Existence[Activity BJ, Activity B] | |	0.0754	0.9947229551451188

In [13]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    # "Responded Existence[Activity BZ, Activity A] | |", # Gigantic
    # "Responded Existence[Activity Q, Activity O] | |", # wide
    "Responded Existence[Accepted+Wait, Completed+Closed] | |", # bpic13
    ]

In [14]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[Accepted+Wait, Completed+Closed] | |    379
dtype: int64
[1, 2, 3, 4, 5, 6, 8, 10, 12, 17, 18, 19, 20, 22, 23, 30, 32, 36, 43, 44, 45, 46, 49, 50, 51, 52, 55, 56, 57, 63, 71, 72, 76, 77, 82, 86, 87, 95, 100, 104, 112, 113, 114, 116, 118, 119, 121, 124, 127, 129, 133, 134, 136, 142, 144, 147, 149, 150, 151, 152, 153, 155, 157, 167, 169, 170, 172, 175, 187, 193, 194, 196, 197, 199, 200, 206, 208, 212, 214, 215, 219, 221, 222, 223, 224, 227, 228, 230, 233, 234, 237, 239, 240, 242, 243, 244, 245, 254, 257, 268, 276, 280, 283, 285, 291, 296, 297, 302, 308, 310, 317, 319, 322, 332, 333, 334, 336, 340, 344, 345, 346, 353, 355, 358, 359, 372, 378, 383, 384, 386, 387, 388, 389, 390, 391, 396, 401, 402, 405, 407, 408, 409, 410, 413, 416, 419, 421, 426, 427, 428, 429, 435, 438, 444, 447, 448, 450, 452, 456, 463, 465, 473, 476, 478, 479, 496, 500, 506, 517, 519, 522, 538, 539, 541, 542, 544, 546, 548, 551, 552, 558, 559, 573, 577, 581, 582, 587, 592, 593, 597, 599, 604, 618, 61

In [15]:
print("END")

END


In [16]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)